# Rosati — UMI-threshold diagnostic

**Why two UMI versions?** Rosati keeps molecular counts (UMI), so abundance is real — but ~90% of
clonotypes are singletons (UMI=1). Filtering at UMI≥2 removes them, giving cleaner but ~10× smaller
clouds. Rather than choose blindly, both thresholds are produced and compared.

These diagnostics show, across all 209 donors: how singleton-dominated the repertoires are, how much
depth and UMI-mass survive the UMI≥2 filter, and how many α–β pairs remain at each threshold. They
justify running both versions (the downstream chain results are identical across thresholds).

# Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# usa 'ss' (DataFrame ya calculado sobre los 209 donantes)
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
colors = {'TRA': '#2C7FB8', 'TRB': '#C0392B'}

# --- Panel A: distribucion del % de singletons por nube ---
ax = axes[0, 0]
for ch in ['TRA', 'TRB']:
    sub = ss[ss['chain'] == ch]
    ax.hist(sub['pct_singleton'], bins=30, alpha=0.6, color=colors[ch],
            label=f'{ch} (median {sub["pct_singleton"].median():.1f}%)', edgecolor='white')
ax.set_xlabel('% singletons (UMI=1) per cloud', fontsize=12, fontweight='bold')
ax.set_ylabel('n donors', fontsize=12, fontweight='bold')
ax.set_title('A) How singleton-dominated is each repertoire?', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.25)

# --- Panel B: profundidad total (UMI>=1) vs filtrada (UMI>=2), por cadena ---
ax = axes[0, 1]
x = np.arange(2); w = 0.35
for i, ch in enumerate(['TRA', 'TRB']):
    sub = ss[ss['chain'] == ch]
    med_total = sub['n_total'].median()
    med_ge2 = sub['n_ge2'].median()
    ax.bar(i - w/2, med_total, w, color=colors[ch], alpha=0.9, label='UMI\u22651 (all)' if i==0 else None)
    ax.bar(i + w/2, med_ge2, w, color=colors[ch], alpha=0.45, hatch='///', label='UMI\u22652' if i==0 else None)
    ax.annotate(f'{med_total:.0f}', (i - w/2, med_total), ha='center', va='bottom', fontsize=9)
    ax.annotate(f'{med_ge2:.0f}', (i + w/2, med_ge2), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['TRA (\u03b1)', 'TRB (\u03b2)'])
ax.set_ylabel('median clonotypes per cloud', fontsize=12, fontweight='bold')
ax.set_title('B) Median cloud depth: all vs UMI\u22652', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.25, axis='y')

# --- Panel C: distribucion de clonotipos retenidos con UMI>=2 (la nube resultante) ---
ax = axes[1, 0]
for ch in ['TRA', 'TRB']:
    sub = ss[ss['chain'] == ch]
    ax.hist(sub['n_ge2'], bins=30, alpha=0.6, color=colors[ch],
            label=f'{ch} (median {sub["n_ge2"].median():.0f})', edgecolor='white')
ax.axvline(100, color='black', ls='--', lw=1.2, alpha=0.7, label='100 clonotypes (min usable?)')
ax.set_xlabel('n clonotypes per cloud after UMI\u22652', fontsize=12, fontweight='bold')
ax.set_ylabel('n donors', fontsize=12, fontweight='bold')
ax.set_title('C) Resulting cloud size with UMI\u22652', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.25)

# --- Panel D: % masa molecular retenida con UMI>=2 ---
ax = axes[1, 1]
for ch in ['TRA', 'TRB']:
    sub = ss[ss['chain'] == ch]
    ax.hist(sub['mass_ge2_pct'], bins=30, alpha=0.6, color=colors[ch],
            label=f'{ch} (median {sub["mass_ge2_pct"].median():.1f}%)', edgecolor='white')
ax.set_xlabel('% molecular mass retained by UMI\u22652', fontsize=12, fontweight='bold')
ax.set_ylabel('n donors', fontsize=12, fontweight='bold')
ax.set_title('D) Signal (UMI mass) kept after filtering singletons', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.25)

plt.suptitle('Rosati UMI-threshold diagnostic across all 209 donors', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# How many complete alpha-beta PAIRS survive each UMI threshold?
# (a pair = donor with BOTH TRA and TRB clouds above a minimum size)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MIN_SIZES = [50, 100, 200, 500]   # minimum clonotypes to call a cloud "usable"

def pairs_surviving(min_clono, use_ge2):
    col = 'n_ge2' if use_ge2 else 'n_total'
    wide = ss.pivot_table(index='donor', columns='chain', values=col).dropna()
    ok = (wide['TRA'] >= min_clono) & (wide['TRB'] >= min_clono)
    return int(ok.sum())

print(f"{'min_clono':>10} | {'pairs UMI>=1':>13} | {'pairs UMI>=2':>13} | {'lost to >=2':>12}")
print("-" * 58)
rows = []
for m in MIN_SIZES:
    n1 = pairs_surviving(m, use_ge2=False)
    n2 = pairs_surviving(m, use_ge2=True)
    rows.append((m, n1, n2))
    print(f"{m:>10} | {n1:>13} | {n2:>13} | {n1-n2:>12}")

rows = np.array(rows)
x = np.arange(len(MIN_SIZES)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(x - w/2, rows[:,1], w, color='#2C7FB8', label='UMI\u22651 (keep singletons)', edgecolor='white')
ax.bar(x + w/2, rows[:,2], w, color='gray', label='UMI\u22652 (drop singletons)', edgecolor='white', hatch='///')
for xi, (n1, n2) in zip(x, rows[:,1:]):
    ax.annotate(str(n1), (xi - w/2, n1), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.annotate(str(n2), (xi + w/2, n2), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(209, color='black', ls=':', lw=1, alpha=0.6)
ax.text(len(MIN_SIZES)-1, 210, 'max = 209 donors', ha='right', fontsize=9, color='black')
ax.set_xticks(x); ax.set_xticklabels([f'\u2265{m}\nclonotypes' for m in MIN_SIZES])
ax.set_xlabel('Minimum cloud size required (both chains)', fontsize=12, fontweight='bold')
ax.set_ylabel('Complete \u03b1-\u03b2 pairs surviving', fontsize=12, fontweight='bold')
ax.set_title('How many alpha-beta pairs survive each UMI threshold?\n(the n available for the pairing experiment)', fontsize=12)
ax.legend(fontsize=10, loc='lower left'); ax.grid(True, alpha=0.25, axis='y')
ax.set_ylim(0, 220)
plt.tight_layout(); plt.show()